In [ ]:
import pandas as pd
import numpy as np

In [ ]:
nombre_archivo = '/content/basededatos.csv'
df = pd.read_csv(nombre_archivo)

In [ ]:
if 'folio 1' in df.columns:
    df = df.drop(columns=['folio 1'])

In [ ]:
text_cols = ['folio', 'Empresa', 'RFC', 'Cliente', 'Status', 'Estatus_Cobranza', 'loan_name', 'Frecuencia']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()
        df[col] = df[col].replace({'NAN': np.nan, 'NONE': np.nan, '': np.nan})

In [ ]:
def limpiar_numero(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    val_str = str(val).replace('$', '').replace(',', '').replace(' ', '').strip()
    try:
        return float(val_str)
    except ValueError:
        return np.nan

num_cols = ['Monto', 'Cuota', 'Plazo', 'Cuotas', 'Tasa_con_impuesto', 'Tasa_sin_impuesto']
for col in num_cols:
    if col in df.columns:
        df[col] = df[col].apply(limpiar_numero)

In [ ]:
date_cols = ['Fecha_Deposito', 'Solicitud', 'Fecha_de_primer_pago']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='mixed', errors='coerce')

In [ ]:
df['Registro_Valido'] = True
df['Motivo_Inconsistencia'] = ""

falta_fecha = df['Fecha_Deposito'].isna()
falta_monto = df['Monto'].isna() | (df['Monto'] <= 0)
falta_id = df['RFC'].isna() & df['Cliente'].isna()

df.loc[falta_fecha | falta_monto | falta_id, 'Registro_Valido'] = False
df.loc[falta_fecha, 'Motivo_Inconsistencia'] += "Sin Fecha Deposito; "
df.loc[falta_monto, 'Motivo_Inconsistencia'] += "Sin Monto Valido; "
df.loc[falta_id, 'Motivo_Inconsistencia'] += "Sin Identificador Cliente/RFC; "

In [ ]:
uma_dict = {
    2021: 89.62,
    2022: 96.22,
    2023: 103.74,
    2024: 108.57,
    2025: 113.14,
    2026: 117.31
}
UMBRAL_AVISO_UMAS = 1605.0

df['Anio_Operacion'] = df['Fecha_Deposito'].dt.year
df['Mes_Operacion'] = df['Fecha_Deposito'].dt.to_period('M').astype(str)
df['UMA_Diaria'] = df['Anio_Operacion'].map(uma_dict).fillna(117.31)
df['Monto_UMAs'] = df['Monto'] / df['UMA_Diaria']

df['Acumulado_6M_Pesos'] = 0.0
df['Acumulado_6M_UMAs'] = 0.0
df['Porcentaje_Umbral_Aviso'] = 0.0

In [ ]:
id_col = 'RFC' if df['RFC'].notna().any() else 'Cliente'
df_validos = df[df['Registro_Valido']].sort_values(by=[id_col, 'Fecha_Deposito']).copy()

acum_pesos = []
acum_umas = []

for _, row in df_validos.iterrows():
    cliente_actual = row[id_col]
    fecha_actual = row['Fecha_Deposito']
    fecha_inicio_6m = fecha_actual - pd.Timedelta(days=180)

    ventana = df_validos[
        (df_validos[id_col] == cliente_actual) &
        (df_validos['Fecha_Deposito'] <= fecha_actual) &
        (df_validos['Fecha_Deposito'] >= fecha_inicio_6m)
    ]
    acum_pesos.append(ventana['Monto'].sum())
    acum_umas.append(ventana['Monto_UMAs'].sum())

df_validos['Acumulado_6M_Pesos'] = acum_pesos
df_validos['Acumulado_6M_UMAs'] = acum_umas
df_validos['Porcentaje_Umbral_Aviso'] = (df_validos['Acumulado_6M_UMAs'] / UMBRAL_AVISO_UMAS) * 100

df.loc[df_validos.index, 'Acumulado_6M_Pesos'] = df_validos['Acumulado_6M_Pesos']
df.loc[df_validos.index, 'Acumulado_6M_UMAs'] = df_validos['Acumulado_6M_UMAs']
df.loc[df_validos.index, 'Porcentaje_Umbral_Aviso'] = df_validos['Porcentaje_Umbral_Aviso']

In [ ]:
es_invalido = (~df['Registro_Valido'].astype(bool)).values
es_rojo = (df['Acumulado_6M_UMAs'].astype(float) >= UMBRAL_AVISO_UMAS).values
es_naranja = (df['Acumulado_6M_UMAs'].astype(float) >= (UMBRAL_AVISO_UMAS * 0.80)).values
es_amarillo = (df['Acumulado_6M_UMAs'].astype(float) >= (UMBRAL_AVISO_UMAS * 0.50)).values

condiciones = [es_invalido, es_rojo, es_naranja, es_amarillo]
etiquetas = [
    'GRIS - REVISIÓN AUDITORÍA / INCONSISTENTE',
    'ROJO - REQUIERE AVISO SAT (>= 1,605 UMAs)',
    'NARANJA - CRÍTICO (>= 80% Umbral)',
    'AMARILLO - PREVENTIVO (>= 50% Umbral)'
]
df['Nivel_Alerta'] = np.select(condiciones, etiquetas, default='VERDE - NORMAL (< 50% Umbral)')
df['Requiere_Aviso_SAT'] = df['Acumulado_6M_UMAs'] >= UMBRAL_AVISO_UMAS

In [ ]:
df['Monto_Formateado'] = df['Monto'].apply(lambda x: f"${x:,.2f}" if pd.notna(x) else "$0.00")
df['Acumulado_6M_Pesos_Formateado'] = df['Acumulado_6M_Pesos'].apply(lambda x: f"${x:,.2f}" if pd.notna(x) else "$0.00")

archivo_salida = 'Base_Operaciones_Credito_TEC_Limpia.csv'
df.to_csv(archivo_salida, index=False, encoding='utf-8-sig')
print(f"Proceso concluido. Archivo generado: {archivo_salida}")

Proceso concluido. Archivo generado: Base_Operaciones_Credito_TEC_Limpia.csv
